# 2 · The model, explained

This notebook walks through the equations (full detail in `docs/model.md`), the finite-volume grid, and one time step solved with **bandsolver**, Newman's BAND method.

## Unknowns and equations

At each node there are four unknowns: electrolyte concentration $c$, solid potential $\Phi_1$, electrolyte potential $\Phi_2$ and the lithium concentration in the particles $c_s$. In the porous cathode:

$$
\varepsilon\frac{\partial c}{\partial t} = -\frac{\partial N_+}{\partial x} + \frac{a\,i_n}{F},\qquad
\frac{\partial i_1}{\partial x} = -a\,i_n,\qquad
\frac{\partial i_2}{\partial x} = a\,i_n,\qquad
\varepsilon_{AM}\frac{\partial c_s}{\partial t} = -\frac{a\,i_n}{F}
$$

with the cation flux and the ionic current of a binary 1:1 electrolyte (dilute-solution theory)

$$
N_+ = -\varepsilon\frac{D_+}{\tau}\frac{\partial c}{\partial x} - \varepsilon\frac{u_+}{\tau}F c\frac{\partial \Phi_2}{\partial x},\qquad
i_2 = -\varepsilon F\frac{D_+ - D_-}{\tau}\frac{\partial c}{\partial x} - \varepsilon F^2\frac{u_+ + u_-}{\tau} c\frac{\partial\Phi_2}{\partial x},
$$

$i_1 = -(1-\varepsilon)\sigma\,\partial\Phi_1/\partial x$, and Butler–Volmer kinetics
$i_n = i_0\left[e^{\alpha_a F\eta/RT} - e^{-\alpha_c F\eta/RT}\right]$ with $\eta = \Phi_1 - \Phi_2 - U(\theta)$.
The separator has only transport. At the lithium foil $N_+ = I/F$ and $\Phi_2 = 0$; at the current collector $i_1 = I$.

## Log variables (corrected mode)

The corrected mode solves for $u = \ln(c/c_\text{bulk})$ and the particles' log-odds $s = \ln(\theta/(1-\theta))$, $\theta = c_s/c_{s,\max}$, instead of $c$ and $c_s$. Concentrations then stay positive and $0 < \theta < 1$ by construction, so the electrolyte can run out and particles can fill or empty smoothly, without clipping. The ion fluxes use exponential fitting (Scharfetter–Gummel), and the open-circuit potential has thermodynamic tails at the ends of the lithiation range (`docs/model.md` section 10). `LogModel.conc` and `LogModel.cs` convert a state back to concentrations.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import bandsolver

from lfp_model.logcore import logit, sigmoid
from lfp_model.logmodel import LogModel
from lfp_model.params import Params

p = Params(mode="corrected", C_rate=1.0)
model = LogModel(p)
m = model.mesh
print(f"{p.nj} nodes; separator/cathode interface at node {m.s}")

## The grid

Control volumes of width $\Delta x_j$, with zero-volume nodes at the lithium foil, at the separator/cathode interface and at the current collector. Those three nodes carry the boundary and interface conditions.

In [ ]:
plt.figure(figsize=(8, 1.6))
plt.plot(m.x * 1e4, np.zeros_like(m.x), "|", ms=12)
plt.axvline(p.L_sep * 1e4, color="k", lw=0.8)
plt.xlabel("x [µm]"); plt.yticks([]); plt.title("node positions (separator | cathode)"); plt.show()

## The open-circuit potential

The fitted LiFePO₄ curve is flat over most of the range and bounded at both ends. The thermodynamic tails (deviation D-16) add the ideal-solution terms $\pm(RT/F)\ln\theta$ that dominate only very close to an empty or full particle (outside $10^{-4} < \theta < 1 - 10^{-4}$), so a particle approaches either end asymptotically. Plotted against $s$, the tails are the straight sections at both ends.

In [ ]:
s = np.linspace(-20, 20, 801)
U = model.kin.ocp(np.zeros_like(s), s)
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
ax[0].plot(sigmoid(s), U); ax[0].set_xlabel("θ"); ax[0].set_ylabel("U [V]")
ax[1].plot(s, U); ax[1].set_xlabel("s = ln(θ/(1-θ))"); ax[1].set_ylabel("U [V]")
for a_ in ax: a_.set_ylim(2.8, 4.2)
plt.tight_layout(); plt.show()

## One time step with BAND

Each backward-Euler step is a nonlinear system in block-tridiagonal form, $A_j\,\delta_{j-1} + B_j\,\delta_j + D_j\,\delta_{j+1} = G_j$ with $G = -R(x)$. The electrode assembles the residual and the blocks, and `bandsolver.solve` solves them. `newton_step` repeats this until the update is below the tolerance.

In [ ]:
x0 = model.initial_state()
R, A, B, D = model.el.residual_and_blocks(x0, x0, 1.0, p.i_app)
print("block shapes:", A.shape, B.shape, D.shape, R.shape)
dx = bandsolver.solve(A, B, D, -R)
print("first-iteration update, max |dΦ1| =", np.abs(dx[:, 1]).max(), "V")

history = []
x1 = model.newton_step(x0, 1.0, p.i_app, history=history)
print("Newton converged in", len(history), "iterations; updates:", ", ".join(f"{h:.1e}" for h in history))

## Checking the Jacobian

`bandsolver.check_jacobian` compares the hand-written Jacobian blocks with finite differences of the residual. The residual here includes the time term of the step from `x0`.

In [ ]:
def fill(x):
    R, A, B, D = model.el.residual_and_blocks(x, x0, 1.0, p.i_app)
    return A, B, D, -R

chk = bandsolver.check_jacobian(fill, x1)
print(f"max relative error {chk.max_error:.1e}")

## Inside the cell during a discharge

In [ ]:
from lfp_model.simulate import run

snapshots = {}
for t_s in (60, 1200, 2400, 3300):
    snapshots[t_s] = run(p, max_steps=t_s).final_state
fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))
for t_s, x in snapshots.items():
    ax[0].plot(m.x * 1e4, model.conc(x) * 1e3, label=f"{t_s} s")
    ax[1].plot(m.x[m.s:] * 1e4, x[m.s:, 2] * 1e3)
    ax[2].plot(m.x[m.s:] * 1e4, model.cs(x)[m.s:] / model.cs_max)
ax[0].set_ylabel("c [M]"); ax[1].set_ylabel("Φ₂ [mV]"); ax[2].set_ylabel("θ")
for a_ in ax: a_.set_xlabel("x [µm]")
ax[0].legend(); plt.tight_layout(); plt.show()